# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [44]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [45]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [46]:
pre=len(df)
new=df.drop_duplicates().copy()

dropped_duplicates=pre-len(new)

log('duplicates', 'Dropped duplicate rows', dropped_duplicates)

[duplicates] Dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [47]:
new['price']=new['price'].str.replace('$','', regex=False)
new['price']=pd.to_numeric(new['price']).astype(float)

log('price','Removed $ and changed price to numeric', len(new))

[price] Removed $ and changed price to numeric (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [48]:
new['qty']=pd.to_numeric(new['qty'])

missing=new['qty'].isna().sum()
negative=(new['qty']<0).sum()

new=new[new['qty'].notna()& (new['qty']>=0)].copy()

log('qty missing', "Dropped rows with missing quantity", missing)
log('qty negative', "Dropped rows with negative quantity", negative)

[qty missing] Dropped rows with missing quantity (12 row(s))
[qty negative] Dropped rows with negative quantity (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [49]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse

In [50]:
print(new['item'].value_counts())
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

new['item']=new['item'].map(ITEM_MAP)

log('item', 'Limited item names to one spelling', len(new))


item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] Limited item names to one spelling (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [51]:
print(new['category'].value_counts())

CATEGORY_MAP= {
  'Food':'Food',
  'food':'Food',
  'Merch':'Merch',
  'Apparel':'Apparel',
  'RainGear':'RainGear',
  'rain-gear':'RainGear'
}

changed=new['category'].isin(['food','Apparel','rain-gear']).sum()

new['category']=new['category'].map(CATEGORY_MAP)
log('category', 'Limited item names to one spelling, kept merch and apparel seperate', len(new))

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] Limited item names to one spelling, kept merch and apparel seperate (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [52]:
assert new.duplicated().sum() == 0
assert new['qty'].min() >= 1
assert new['price'].dtype == float
assert new['item'].nunique()==3
assert new['category'].nunique()==4
print('clean:', new.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [53]:
new['revenue']=new['qty']*new['price']

revenue_by_cat=new.groupby('category')['revenue'].sum().sort_values(ascending=False)

print(revenue_by_cat.round(2))
print('Overall Total:', round(new['revenue'].sum(),2))

category
Food        1656.0
RainGear    1512.0
Merch        856.5
Apparel      715.5
Name: revenue, dtype: float64
Overall Total: 4740.0


**What I would tell the vendor: Given that food generated the highest revenue of any category, I'd tell the vendor to stock more food.

### TODO 8 — read back your log

In [54]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,Dropped duplicate rows,15
1,price,Removed $ and changed price to numeric,300
2,qty missing,Dropped rows with missing quantity,12
3,qty negative,Dropped rows with negative quantity,13
4,item,Limited item names to one spelling,275
5,category,"Limited item names to one spelling, kept merch...",275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

In [55]:
check=df.copy()
check['price']=check['price'].str.replace('$','',regex=False).astype(float)
check['revenue']=check['qty']*check['price']
print('Before cleaning:', check['revenue'].sum())

check=check.drop_duplicates()
print('After dropping duplicates:', check['revenue'].sum())

check=check[check['qty'].notna()& (check['qty']>=0)]
print("After cleaning qty:", check['revenue'].sum())

Before cleaning: 4852.5
After dropping duplicates: 4594.5
After cleaning qty: 4740.0


a) The cleaning step that changed revenue the most was dropping duplicate rows. The revenue total before that step was $4852.5, and after it was $4594.5, so roughly a $250 shift.



b) I chose to keep merch and apparel as seperate categories, while another person may have chosen to combine them. Combining them wouldn't have changed the total revenue, but it would have made their joint category appear as the second highest revenue category, coming in at $1572. I chose to keep the cateogries seperate because they are not exactly the same thing, and I'd usually prefer to have more detail in the data rather than combining categories.